<a href="https://colab.research.google.com/github/marianovescio/Gittest/blob/main/MarianoVescio_TP1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalación de librerías

In [52]:
!pip install deltalake pyarrow pandas requests

## Importación de librerías

In [53]:
import os
from datetime import datetime

import pandas as pd
import requests
from deltalake.writer import write_deltalake

# MarianoVescio_TP1
**Extracción y almacenamiento de datos**

### Objetivo
Desarrollar un proceso de extracción de datos desde una API utilizando Python, convertir la respuesta a DataFrames de Pandas y almacenar los datos en formato Delta Lake.

### API seleccionada
Se utilizará la API de CoinGecko, tomando:
- un endpoint dinámico: `coins/markets`
- un endpoint estático: `coins/list`

### Estrategia de extracción
- **Full extraction** para `coins/list`, por tratarse de una tabla de referencia relativamente estable.
- **Incremental extraction** para `coins/markets`, ya que contiene datos temporales que cambian periódicamente.

### Estrategia de almacenamiento
Los datos se almacenarán en una estructura tipo data lake, en formato Delta Lake, separando datasets en distintos directorios.

## Definición de rutas del data lake

In [54]:
BASE_PATH = "data_lake/bronze"
COINS_LIST_PATH = f"{BASE_PATH}/coins_list"
COINS_MARKETS_PATH = f"{BASE_PATH}/coins_markets"

# =========================
# CONFIGURACIÓN MINIO
# =========================

MINIO_BUCKET = "marianovescio-bucket"
MINIO_ENDPOINT = "http://31.97.241.212:9002"
MINIO_ACCESS_KEY = "marianovescio"
MINIO_SECRET_KEY = "marianovescio"

MINIO_COINS_LIST_PATH = f"s3://{MINIO_BUCKET}/bronze/coins_list"
MINIO_COINS_MARKETS_PATH = f"s3://{MINIO_BUCKET}/bronze/coins_markets"

storage_options = {
    "AWS_ACCESS_KEY_ID": MINIO_ACCESS_KEY,
    "AWS_SECRET_ACCESS_KEY": MINIO_SECRET_KEY,
    "AWS_ENDPOINT_URL": MINIO_ENDPOINT,
    "AWS_REGION": "us-east-1",
    "AWS_ALLOW_HTTP": "true",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_S3_USE_SSL": "false"
}

In [55]:
def ensure_directory(path: str):
    """
    Crea un directorio si no existe.
    """
    os.makedirs(path, exist_ok=True)

In [56]:
ensure_directory(COINS_LIST_PATH)
ensure_directory(COINS_MARKETS_PATH)

print("Directorios creados correctamente.")
print(COINS_LIST_PATH)
print(COINS_MARKETS_PATH)

Directorios creados correctamente.
data_lake/bronze/coins_list
data_lake/bronze/coins_markets


## Extracción full - Endpoint coins/list

In [57]:
def extract_coins_list():
    """
    Realiza una extracción full del endpoint /coins/list de CoinGecko.
    Retorna un DataFrame de Pandas.
    """
    url = "https://api.coingecko.com/api/v3/coins/list"

    response = requests.get(url, timeout=30)
    response.raise_for_status()  # lanza error si la request falla

    data = response.json()
    df = pd.DataFrame(data)

    return df

In [58]:
df_coins_list = extract_coins_list()

print("Cantidad de registros:", len(df_coins_list))
print("Columnas:", df_coins_list.columns.tolist())

df_coins_list.head()

Cantidad de registros: 18025
Columnas: ['id', 'symbol', 'name']


,id,symbol,name
0,_,gib,༼ つ ◕_◕ ༽つ
1,000-capital,000,000 Capital
2,01111010011110000110001001110100-token,01111010011110000110001001110100,01111010011110000110001001110100
3,01112131415161718192021222324252-token,1234567891,01112131415161718192021222324252
4,01-token,01,01


In [59]:
extraction_datetime = datetime.now()

df_coins_list["extraction_timestamp"] = extraction_datetime
df_coins_list["extraction_date"] = extraction_datetime.date().isoformat()

Almacenamiento Full en Data Lake

In [60]:
write_deltalake(
    COINS_LIST_PATH,
    df_coins_list,
    mode="overwrite"
)

print(f"Datos guardados correctamente en: {COINS_LIST_PATH}")

#Guardado en MINIO#
write_deltalake(
    MINIO_COINS_LIST_PATH,
    df_coins_list,
    mode="overwrite",
    storage_options=storage_options
)

print(f"Datos guardados también en MinIO: {MINIO_COINS_LIST_PATH}")

Datos guardados correctamente en: data_lake/bronze/coins_list
Datos guardados también en MinIO: s3://marianovescio-bucket/bronze/coins_list


In [61]:
print("Extracción full de coins_list finalizada correctamente.")
print(f"Total de registros guardados: {len(df_coins_list)}")

Extracción full de coins_list finalizada correctamente.
Total de registros guardados: 18025


## Extracción incremental - Endpoint coins/markets

In [62]:
def extract_coins_markets(vs_currency="usd", per_page=50, page=1):
    """
    Extrae datos del endpoint /coins/markets (datos dinámicos).
    """
    url = "https://api.coingecko.com/api/v3/coins/markets"

    params = {
        "vs_currency": vs_currency,
        "order": "market_cap_desc",
        "per_page": per_page,
        "page": page
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data)

    return df

In [63]:
df_coins_markets = extract_coins_markets()

print("Cantidad de registros:", len(df_coins_markets))
df_coins_markets.head()

Cantidad de registros: 50


,id,symbol,name,image,current_price,market_cap,market_cap_rank,fully_diluted_valuation,total_volume,high_24h,...,total_supply,max_supply,ath,ath_change_percentage,ath_date,atl,atl_change_percentage,atl_date,roi,last_updated
0,bitcoin,btc,Bitcoin,https://coin-images.coingecko.com/coins/images...,70447.000000,1411278585199,1,1411278585199,5.211121e+10,71646.000000,...,2.000304e+07,2.100000e+07,126080.00,-44.12541,2025-10-06T18:57:42.558Z,67.810000,1.037898e+05,2013-07-06T00:00:00.000Z,None,2026-03-24T02:16:24.557Z
1,ethereum,eth,Ethereum,https://coin-images.coingecko.com/coins/images...,2135.440000,258217324565,2,258217324565,2.718246e+10,2186.900000,...,1.206916e+08,NaN,4946.05,-56.82532,2025-08-24T19:21:03.333Z,0.432979,4.930974e+05,2015-10-20T00:00:00.000Z,"{'times': 39.52545459914526, 'currency': 'btc'...",2026-03-24T02:16:24.699Z
2,tether,usdt,Tether,https://coin-images.coingecko.com/coins/images...,0.999717,184141018390,3,189606701237,8.963107e+10,0.999836,...,1.896503e+11,NaN,1.32,-24.43694,2018-07-24T00:00:00.000Z,0.572521,7.462613e+01,2015-03-02T00:00:00.000Z,None,2026-03-24T02:16:00.133Z
3,ripple,xrp,XRP,https://coin-images.coingecko.com/coins/images...,1.410000,86661215880,4,141249339509,3.333685e+09,1.460000,...,9.998570e+10,1.000000e+11,3.65,-61.28071,2025-07-18T03:40:53.808Z,0.002686,5.245931e+04,2014-05-22T00:00:00.000Z,None,2026-03-24T02:16:24.335Z
4,binancecoin,bnb,BNB,https://coin-images.coingecko.com/coins/images...,632.550000,86439857928,5,86439857928,1.646148e+09,648.310000,...,1.363575e+08,2.000000e+08,1369.99,-53.82802,2025-10-13T08:41:24.131Z,0.039818,1.588521e+06,2017-10-19T00:00:00.000Z,None,2026-03-24T02:16:24.266Z


In [64]:
extraction_datetime = datetime.now()

df_coins_markets["extraction_timestamp"] = extraction_datetime
df_coins_markets["extraction_date"] = extraction_datetime.date().isoformat()
df_coins_markets["extraction_hour"] = extraction_datetime.strftime("%H")

## Almacenamiento incremental en Delta Lake

In [65]:
write_deltalake(
    COINS_MARKETS_PATH,
    df_coins_markets,
    mode="append",
    partition_by=["extraction_date", "extraction_hour"]
)

write_deltalake(
    MINIO_COINS_MARKETS_PATH,
    df_coins_markets,
    mode="append",
    partition_by=["extraction_date", "extraction_hour"],
    storage_options=storage_options
)

print(f"Datos guardados en MinIO: {MINIO_COINS_MARKETS_PATH}")

Datos guardados en MinIO: s3://marianovescio-bucket/bronze/coins_markets


In [66]:
print("Extracción incremental finalizada correctamente.")
print(f"Total de registros guardados: {len(df_coins_markets)}")

Extracción incremental finalizada correctamente.
Total de registros guardados: 50
